In [1]:
# ============================================================
# GIÁO TRÌNH: THỊ GIÁC MÁY: TỪ XỬ LÝ ẢNH ĐẾN HỌC SÂU
# BÀI CODE MINH HỌA: KHẢO SÁT ATTENTION ROLLOUT VÀ GRAD-CAM THEO TOKEN GRID
# Chương/Mục liên quan: Chương 8 - Mô hình học sâu cho các tác vụ thị giác máy
# ============================================================

# ============================================================
# MÔ TẢ
# ============================================================

# Mục đích:
# - Minh họa cách khảo sát vùng ảnh được các mô hình Transformer thị giác chú ý.
# - So sánh Attention Rollout và Grad-CAM trên ViT, Swin Transformer và MaxViT.
# - Quan sát sự thay đổi bản đồ chú ý khi ảnh gốc bị che một phần.

# Sau khi chạy code, người học cần:
# 1. Quan sát được vùng ảnh có ảnh hưởng mạnh đến quyết định phân loại.
# 2. Hiểu cách attention map và Grad-CAM có thể được biểu diễn trên lưới token.
# 3. So sánh phản ứng của mô hình khi che vùng đầu hoặc thân của đối tượng.

# Input:
# - Dữ liệu đầu vào: ảnh màu từ GitHub
# - Kiểu dữ liệu: ảnh RGB
# - Kích thước đầu vào: được tự động resize theo yêu cầu của từng mô hình timm

# Output:
# - Ảnh gốc
# - Ảnh che đầu
# - Ảnh che thân
# - Attention Rollout token grid
# - Grad-CAM token grid
# - Bảng tóm tắt nhãn dự đoán, độ tin cậy, kích thước grid và số khối attention

# Lưu ý:
# Đoạn code này được xây dựng với sự hỗ trợ của công cụ AI.
# Giảng viên đã đọc, kiểm tra và hiệu chỉnh nhằm bảo đảm tính chính xác,
# tính sư phạm và sự phù hợp với nội dung lý thuyết trong giáo trình.

# ============================================================
# 1. CÀI ĐẶT VÀ IMPORT THƯ VIỆN
# ============================================================

!pip install grad-cam -q
!pip install timm -q

import math
import requests
from io import BytesIO

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
import torch.nn.functional as F
import timm
from timm.data import resolve_model_data_config, create_transform

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


# ============================================================
# 2. CẤU HÌNH THAM SỐ
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_URL = "https://github.com/lthavnu/cv-book/raw/main/images/samoyed.jpg"

MODEL_SPECS = {
    "ViT": "vit_base_patch16_224",
    "Swin": "swin_tiny_patch4_window7_224",
    "MaxViT": "maxvit_tiny_tf_224",
}

try:
    if hasattr(timm.layers, "set_fused_attn"):
        timm.layers.set_fused_attn(False)
        print("[INFO] fused attention = OFF")
except Exception as e:
    print("[WARN] Không tắt được fused attention:", e)


# ============================================================
# 3. TẢI DỮ LIỆU ĐẦU VÀO
# ============================================================

def load_image_from_url(url):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGB")


# ============================================================
# 4. CÁC HÀM TIỆN ÍCH
# ============================================================

def normalize01(x):
    x = x.astype(np.float32)
    x = x - x.min()
    if x.max() > 1e-8:
        x = x / x.max()
    return x


def softmax_confidence(logits):
    prob = torch.softmax(logits, dim=-1)
    conf, idx = prob.max(dim=-1)
    return idx.item(), conf.item()


def prepare_input(model, pil_img):
    config = resolve_model_data_config(model)
    transform = create_transform(**config, is_training=False)
    x = transform(pil_img).unsqueeze(0)
    return x


def pil_to_np01(pil_img):
    return np.asarray(pil_img).astype(np.float32) / 255.0


# ============================================================
# 5. TẠO ẢNH BỊ CHE MỘT PHẦN
# ============================================================

def create_occluded_image(pil_img, mode="head", fill=(128, 128, 128), border=True):
    img = pil_img.copy()
    draw = ImageDraw.Draw(img)
    W, H = img.size

    if mode == "head":
        x1 = int(0.06 * W)
        y1 = int(0.08 * H)
        x2 = int(0.29 * W)
        y2 = int(0.34 * H)
    elif mode == "body":
        x1 = int(0.20 * W)
        y1 = int(0.26 * H)
        x2 = int(0.50 * W)
        y2 = int(0.60 * H)
    else:
        raise ValueError("mode phải là 'head' hoặc 'body'")

    draw.rectangle([x1, y1, x2, y2], fill=fill)

    if border:
        draw.rectangle([x1, y1, x2, y2], outline=(255, 0, 0), width=3)

    return img, (x1, y1, x2, y2)


# ============================================================
# 6. THU ATTENTION TỪ MÔ HÌNH
# ============================================================

class AttentionCollector:
    def __init__(self, model):
        self.records = []
        self.hooks = []

        candidates = []
        for name, module in model.named_modules():
            lname = name.lower()
            if "attn_drop" in lname or "drop_attn" in lname:
                candidates.append((name, module))

        if len(candidates) == 0:
            raise RuntimeError(
                "Không tìm thấy module attn_drop nào. "
                "Có thể phiên bản timm của bạn dùng tên khác."
            )

        def hook_fn(name):
            def _fn(module, inputs, output):
                if torch.is_tensor(output):
                    self.records.append((name, output.detach().cpu()))
            return _fn

        for name, module in candidates:
            self.hooks.append(module.register_forward_hook(hook_fn(name)))

    def clear(self):
        self.records.clear()

    def close(self):
        for h in self.hooks:
            h.remove()
        self.hooks.clear()


# ============================================================
# 7. TÍNH ATTENTION ROLLOUT
# ============================================================

def vit_extract_layer_mats(attn_records):
    mats = []

    for name, attn in attn_records:
        if attn.ndim != 4:
            continue

        # Công thức liên hệ lý thuyết:
        # Lấy trung bình attention theo các head để thu ma trận attention của một lớp.
        a = attn[0].mean(dim=0).numpy()
        mats.append((name, a))

    return mats


def vit_attention_rollout(attn_mats):
    result = None

    for _, A in attn_mats:
        A = A.copy()
        I = np.eye(A.shape[0], dtype=np.float32)

        # Công thức liên hệ lý thuyết:
        # A_hat = A + I, sau đó chuẩn hóa theo hàng.
        A = A + I
        A = A / (A.sum(axis=-1, keepdims=True) + 1e-8)

        # Công thức liên hệ lý thuyết:
        # Nhân liên tiếp các ma trận attention qua các lớp.
        result = A if result is None else (A @ result)

    cls_to_patch = result[0, 1:]
    side = int(math.sqrt(cls_to_patch.shape[0]))
    heat = cls_to_patch.reshape(side, side)

    return normalize01(heat)


def hierarchical_layer_map(attn_tensor):
    if attn_tensor.ndim != 4:
        return None

    # Công thức liên hệ lý thuyết:
    # Trung bình attention theo batch/window và theo head để thu độ quan trọng token.
    a = attn_tensor.mean(dim=0)
    a = a.mean(dim=0)
    token_importance = a.mean(dim=0).numpy()

    T = token_importance.shape[0]
    side = int(math.sqrt(T))

    if side * side != T:
        return None

    heat = token_importance.reshape(side, side)

    return normalize01(heat)


def resize_grid_np(x2d, out_h, out_w):
    t = torch.tensor(x2d, dtype=torch.float32)[None, None, ...]
    t = F.interpolate(t, size=(out_h, out_w), mode="bilinear", align_corners=False)
    return t[0, 0].cpu().numpy()


def hierarchical_rollout_like(attn_records, target_side=14):
    acc = np.ones((target_side, target_side), dtype=np.float32)

    for _, attn in attn_records:
        h = hierarchical_layer_map(attn)

        if h is None:
            continue

        h = resize_grid_np(h, target_side, target_side)
        h = normalize01(h)

        # Công thức liên hệ lý thuyết:
        # Tích lũy bản đồ chú ý qua nhiều lớp theo dạng rollout xấp xỉ.
        acc = 0.5 * acc + 0.5 * (acc * (h + 1e-6))
        acc = normalize01(acc)

    return normalize01(acc)


# ============================================================
# 8. CHỌN LỚP ĐÍCH CHO GRAD-CAM
# ============================================================

def pick_gradcam_target_layer(model, arch_name):
    if arch_name == "ViT":
        return model.blocks[-1].norm1

    elif arch_name == "Swin":
        return model.layers[-1].blocks[-1].norm1

    elif arch_name == "MaxViT":
        candidates = [
            lambda m: m.stages[-1].blocks[-1].norm1,
            lambda m: m.stages[-1].blocks[-1].attn_block.norm1,
        ]

        for fn in candidates:
            try:
                return fn(model)
            except Exception:
                pass

        last_norm1 = None

        for name, module in model.named_modules():
            if name.lower().endswith("norm1"):
                last_norm1 = module

        if last_norm1 is not None:
            return last_norm1

        raise RuntimeError("Không tìm được target layer phù hợp cho MaxViT.")

    else:
        raise ValueError(f"Unsupported arch_name: {arch_name}")


# ============================================================
# 9. RESHAPE TRANSFORM CHO GRAD-CAM
# ============================================================

def make_reshape_transform(arch_name):
    def reshape_transform(tensor):
        if tensor.ndim == 3:
            B, N, C = tensor.shape

            if arch_name == "ViT":
                tensor = tensor[:, 1:, :]
                N = tensor.shape[1]

            side = int(math.sqrt(N))

            if side * side != N:
                raise RuntimeError(
                    f"{arch_name}: không reshape được N={N} thành lưới vuông."
                )

            tensor = tensor.reshape(B, side, side, C)
            tensor = tensor.permute(0, 3, 1, 2).contiguous()

            return tensor

        elif tensor.ndim == 4:
            if tensor.shape[-1] > max(tensor.shape[1], tensor.shape[2]):
                tensor = tensor.permute(0, 3, 1, 2).contiguous()

            return tensor

        else:
            raise RuntimeError(
                f"{arch_name}: tensor ndim={tensor.ndim} không hỗ trợ reshape_transform."
            )

    return reshape_transform


# ============================================================
# 10. TÍNH GRAD-CAM
# ============================================================

def compute_gradcam(model, arch_name, pil_img, target_category=None):
    x = prepare_input(model, pil_img).to(DEVICE)

    with torch.no_grad():
        logits = model(x)

    pred_idx, conf = softmax_confidence(logits)

    if target_category is None:
        target_category = pred_idx

    target_layers = [pick_gradcam_target_layer(model, arch_name)]
    reshape_transform = make_reshape_transform(arch_name)

    cam = GradCAM(
        model=model,
        target_layers=target_layers,
        reshape_transform=reshape_transform
    )

    targets = [ClassifierOutputTarget(target_category)]

    # Thuật toán liên hệ lý thuyết:
    # Grad-CAM dùng gradient của điểm phân loại theo đặc trưng tại lớp đích
    # để tạo bản đồ vùng ảnh đóng góp mạnh cho dự đoán.
    grayscale_cam = cam(input_tensor=x, targets=targets)[0]
    grayscale_cam = normalize01(grayscale_cam)

    return {
        "pred_idx": pred_idx,
        "conf": conf,
        "cam": grayscale_cam,
    }


# ============================================================
# 11. CHUYỂN GRAD-CAM THÀNH TOKEN GRID
# ============================================================

def heatmap_to_token_grid(heatmap_2d, grid_h, grid_w):
    H, W = heatmap_2d.shape
    cell_h = H / grid_h
    cell_w = W / grid_w

    out = np.zeros((grid_h, grid_w), dtype=np.float32)

    for i in range(grid_h):
        for j in range(grid_w):
            y1 = int(round(i * cell_h))
            y2 = int(round((i + 1) * cell_h))
            x1 = int(round(j * cell_w))
            x2 = int(round((j + 1) * cell_w))

            patch = heatmap_2d[y1:y2, x1:x2]

            if patch.size == 0:
                out[i, j] = 0.0
            else:
                # Công thức liên hệ lý thuyết:
                # Giá trị token được lấy bằng trung bình Grad-CAM trong vùng ảnh tương ứng.
                out[i, j] = patch.mean()

    return normalize01(out)


# ============================================================
# 12. HÀM HIỂN THỊ TOKEN GRID
# ============================================================

def draw_token_grid_overlay(
    ax,
    rgb_img,
    token_grid,
    title="",
    alpha=0.50,
    cmap_name="jet",
    grid_linewidth=1.0
):
    H, W = rgb_img.shape[:2]
    gh, gw = token_grid.shape

    ax.imshow(rgb_img)
    cmap = plt.get_cmap(cmap_name)

    cell_h = H / gh
    cell_w = W / gw

    for i in range(gh):
        for j in range(gw):
            val = float(token_grid[i, j])
            color = cmap(val)

            rect = plt.Rectangle(
                (j * cell_w, i * cell_h),
                cell_w,
                cell_h,
                facecolor=color,
                edgecolor=(1, 1, 1, 0.35),
                linewidth=grid_linewidth,
                alpha=alpha
            )

            ax.add_patch(rect)

    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.set_title(title)
    ax.axis("off")


def draw_token_grid_only(
    ax,
    token_grid,
    title="",
    cmap_name="jet",
    show_values=False
):
    gh, gw = token_grid.shape

    ax.imshow(token_grid, cmap=cmap_name, interpolation="nearest", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xticks(np.arange(gw))
    ax.set_yticks(np.arange(gh))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.grid(color="white", linestyle="-", linewidth=0.8)

    if show_values:
        for i in range(gh):
            for j in range(gw):
                ax.text(
                    j,
                    i,
                    f"{token_grid[i, j]:.2f}",
                    ha="center",
                    va="center",
                    fontsize=6,
                    color="black"
                )

    ax.tick_params(length=0)


# ============================================================
# 13. XỬ LÝ CHÍNH CHO MỘT ẢNH
# ============================================================

def run_one_pass(model, arch_name, pil_img):
    x = prepare_input(model, pil_img).to(DEVICE)
    collector = AttentionCollector(model)

    with torch.no_grad():
        collector.clear()
        logits = model(x)

    pred_idx, conf = softmax_confidence(logits)
    attn_records = collector.records
    collector.close()

    if arch_name == "ViT":
        mats = vit_extract_layer_mats(attn_records)
        rollout_grid = vit_attention_rollout(mats)
    else:
        rollout_grid = hierarchical_rollout_like(attn_records, target_side=14)

    gc = compute_gradcam(model, arch_name, pil_img, target_category=pred_idx)
    gradcam_heatmap = gc["cam"]

    gh, gw = rollout_grid.shape
    gradcam_grid = heatmap_to_token_grid(gradcam_heatmap, gh, gw)

    return {
        "pred_idx": pred_idx,
        "conf": conf,
        "rollout_grid": rollout_grid,
        "gradcam_heatmap": gradcam_heatmap,
        "gradcam_grid": gradcam_grid,
        "num_attn_blocks": len(attn_records),
    }


# ============================================================
# 14. CHẠY CHO TỪNG MÔ HÌNH
# ============================================================

def run_model_cases(model_name, arch_name, images_dict):
    print(f"\n================ {arch_name} =================")
    print(f"[INFO] Loading {model_name}")

    model = timm.create_model(model_name, pretrained=True).to(DEVICE).eval()

    out = {}

    for case_name, pil_img in images_dict.items():
        out[case_name] = run_one_pass(model, arch_name, pil_img)

    return out


# ============================================================
# 15. TẢI ẢNH, TẠO ẢNH CHE VÀ CHẠY THỰC NGHIỆM
# ============================================================

img_clean = load_image_from_url(IMAGE_URL)
img_head, bbox_head = create_occluded_image(img_clean, mode="head")
img_body, bbox_body = create_occluded_image(img_clean, mode="body")

images = {
    "Gốc": img_clean,
    "Che đầu": img_head,
    "Che thân": img_body,
}

img_np = {k: pil_to_np01(v) for k, v in images.items()}

results = {}

for arch_name, model_name in MODEL_SPECS.items():
    results[arch_name] = run_model_cases(model_name, arch_name, images)


# ============================================================
# 16. HIỂN THỊ KẾT QUẢ CHỒNG TOKEN GRID LÊN ẢNH
# ============================================================

fig, axes = plt.subplots(3, 9, figsize=(28, 12))

fig.suptitle(
    "So sánh Attention Rollout và Grad-CAM theo token grid\n"
    "cho ViT / Swin / MaxViT trên ảnh gốc, ảnh che đầu và ảnh che thân",
    fontsize=18
)

row_order = ["ViT", "Swin", "MaxViT"]
case_order = ["Gốc", "Che đầu", "Che thân"]

for r, arch_name in enumerate(row_order):
    for i, case_name in enumerate(case_order):
        res = results[arch_name][case_name]
        rgb = img_np[case_name]

        c0 = i * 3

        axes[r, c0 + 0].imshow(rgb)
        axes[r, c0 + 0].set_title(f"{arch_name} - {case_name}")
        axes[r, c0 + 0].axis("off")

        draw_token_grid_overlay(
            axes[r, c0 + 1],
            rgb,
            res["rollout_grid"],
            title=f"Rollout grid\npred={res['pred_idx']}, conf={res['conf']:.3f}",
            alpha=0.45,
            cmap_name="jet",
            grid_linewidth=0.8
        )

        draw_token_grid_overlay(
            axes[r, c0 + 2],
            rgb,
            res["gradcam_grid"],
            title="Grad-CAM grid",
            alpha=0.45,
            cmap_name="jet",
            grid_linewidth=0.8
        )

plt.tight_layout()
plt.show()


# ============================================================
# 17. HIỂN THỊ TOKEN GRID THUẦN
# ============================================================

fig2, axes2 = plt.subplots(6, 9, figsize=(28, 18))

fig2.suptitle(
    "Bản đồ token grid thuần cho Attention Rollout và Grad-CAM",
    fontsize=18
)

row_defs = [
    ("ViT", "Gốc"),
    ("ViT", "Che đầu"),
    ("ViT", "Che thân"),
    ("Swin", "Gốc"),
    ("Swin", "Che đầu"),
    ("Swin", "Che thân"),
]

for r, (arch_name, case_name) in enumerate(row_defs):
    res = results[arch_name][case_name]

    axes2[r, 0].imshow(img_np[case_name])
    axes2[r, 0].set_title(f"{arch_name} - {case_name}")
    axes2[r, 0].axis("off")

    draw_token_grid_only(axes2[r, 1], res["rollout_grid"], title="Rollout grid")
    draw_token_grid_only(axes2[r, 2], res["gradcam_grid"], title="Grad-CAM grid")

    axes2[r, 3].imshow(img_np[case_name])
    axes2[r, 3].axis("off")
    axes2[r, 3].set_title("Ảnh")

    draw_token_grid_only(axes2[r, 4], res["rollout_grid"], title="Rollout")
    draw_token_grid_only(axes2[r, 5], res["gradcam_grid"], title="Grad-CAM")

    axes2[r, 6].imshow(img_np[case_name])
    axes2[r, 6].axis("off")
    axes2[r, 6].set_title("Ảnh")

    draw_token_grid_only(axes2[r, 7], res["rollout_grid"], title="Rollout")
    draw_token_grid_only(axes2[r, 8], res["gradcam_grid"], title="Grad-CAM")

plt.tight_layout()
plt.show()


fig3, axes3 = plt.subplots(3, 3, figsize=(10, 12))

fig3.suptitle("MaxViT - token grid thuần", fontsize=18)

for r, case_name in enumerate(case_order):
    res = results["MaxViT"][case_name]

    axes3[r, 0].imshow(img_np[case_name])
    axes3[r, 0].set_title(f"MaxViT - {case_name}")
    axes3[r, 0].axis("off")

    draw_token_grid_only(axes3[r, 1], res["rollout_grid"], title="Rollout grid")
    draw_token_grid_only(axes3[r, 2], res["gradcam_grid"], title="Grad-CAM grid")

plt.tight_layout()
plt.show()


# ============================================================
# 18. KIỂM TRA KẾT QUẢ
# ============================================================

print("\n================ KIỂM TRA KẾT QUẢ ================")

assert img_clean.size[0] > 0 and img_clean.size[1] > 0
assert set(results.keys()) == set(row_order)

for arch_name in row_order:
    for case_name in case_order:
        res = results[arch_name][case_name]

        assert "rollout_grid" in res
        assert "gradcam_grid" in res
        assert res["rollout_grid"].ndim == 2
        assert res["gradcam_grid"].ndim == 2
        assert res["rollout_grid"].shape == res["gradcam_grid"].shape
        assert 0.0 <= res["conf"] <= 1.0

print("Ảnh đã tải thành công.")
print("Các mô hình đã chạy thành công.")
print("Attention Rollout grid và Grad-CAM grid có cùng kích thước.")
print("Độ tin cậy dự đoán nằm trong khoảng [0, 1].")


# ============================================================
# 19. IN TÓM TẮT
# ============================================================

print("\n================ TÓM TẮT ================")
print(f"Head bbox: {bbox_head}")
print(f"Body bbox: {bbox_body}")

for arch_name in row_order:
    print(f"\n----- {arch_name} -----")

    for case_name in case_order:
        res = results[arch_name][case_name]
        gh, gw = res["rollout_grid"].shape

        print(
            f"{case_name:8s} | pred={res['pred_idx']:4d} | conf={res['conf']:.4f} "
            f"| grid={gh}x{gw} | attn_blocks={res['num_attn_blocks']}"
        )


# ============================================================
# 20. GỢI Ý THỬ NGHIỆM CHO NGƯỜI HỌC
# ============================================================

# 1. Thay IMAGE_URL bằng ảnh khác để quan sát bản đồ chú ý trên đối tượng khác.
# 2. Thay vị trí che trong create_occluded_image để kiểm tra vùng nào ảnh hưởng mạnh đến dự đoán.
# 3. Thay MODEL_SPECS để so sánh thêm các biến thể Transformer thị giác khác trong thư viện timm.

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import os
import shutil
import zipfile
import matplotlib.pyplot as plt

# Tạo thư mục để lưu ảnh nếu chưa tồn tại
output_dir_no_title = "attention_maps_overlay_no_title"
os.makedirs(output_dir_no_title, exist_ok=True)

image_files_no_title = []

# --- Lưu ảnh gốc, ảnh che đầu, ảnh che thân --- #
for case_name, pil_img in images.items():
    base_image_filename = os.path.join(output_dir_no_title, f"Base_Image_{case_name}.png")
    pil_img.save(base_image_filename)
    image_files_no_title.append(base_image_filename)
    print(f"Đã lưu ảnh cơ sở '{case_name}' vào {base_image_filename}")

for arch_name in row_order:
    for case_name in case_order:
        res = results[arch_name][case_name]
        rgb = img_np[case_name]
        rollout_grid = res['rollout_grid']
        gradcam_grid = res['gradcam_grid']

        # --- Lưu Rollout Overlay (không title, không margin) --- #
        fig_rollout_overlay, ax_rollout_overlay = plt.subplots(1, 1, figsize=(4, 4))
        draw_token_grid_overlay(
            ax_rollout_overlay,
            rgb,
            rollout_grid,
            title="", # Không có tiêu đề
            alpha=0.45,
            cmap_name="jet",
            grid_linewidth=0.0
        )
        rollout_overlay_filename = os.path.join(output_dir_no_title, f"{arch_name}_{case_name}_rollout_overlay_no_title.png")
        plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
        fig_rollout_overlay.savefig(rollout_overlay_filename, dpi=300, bbox_inches='tight',pad_inches=0)
        image_files_no_title.append(rollout_overlay_filename)
        plt.close(fig_rollout_overlay) # Đóng figure để giải phóng bộ nhớ

        # --- Lưu Grad-CAM Overlay (không title, không margin) --- #
        fig_gradcam_overlay, ax_gradcam_overlay = plt.subplots(1, 1, figsize=(4, 4))
        draw_token_grid_overlay(
            ax_gradcam_overlay,
            rgb,
            gradcam_grid,
            title="", # Không có tiêu đề
            alpha=0.45,
            cmap_name="jet",
            grid_linewidth=0.0
        )
        gradcam_overlay_filename = os.path.join(output_dir_no_title, f"{arch_name}_{case_name}_gradcam_overlay_no_title.png")
        fig_gradcam_overlay.savefig(gradcam_overlay_filename, dpi=300, bbox_inches='tight',pad_inches=0)
        image_files_no_title.append(gradcam_overlay_filename)
        plt.close(fig_gradcam_overlay) # Đóng figure để giải phóng bộ nhớ

print(f"Đã lưu {len(image_files_no_title)} hình ảnh overlay (không tiêu đề) vào thư mục: {output_dir_no_title}/")

# Nén tất cả các file ảnh đã tạo vào một file zip
zip_filename_no_title = "attention_maps_overlay_no_title.zip"
with zipfile.ZipFile(zip_filename_no_title, 'w') as zf:
    for file_path in image_files_no_title:
        zf.write(file_path, arcname=os.path.basename(file_path))

print(f"Đã tạo file nén '{zip_filename_no_title}' chứa các hình ảnh overlay không tiêu đề.")
print("Bạn có thể tải file này xuống.")

# Dọn dẹp thư mục sau khi nén
shutil.rmtree(output_dir_no_title)
print(f"Đã xóa thư mục tạm thời: {output_dir_no_title}")

Đã lưu ảnh cơ sở 'Gốc' vào attention_maps_overlay_no_title/Base_Image_Gốc.png
Đã lưu ảnh cơ sở 'Che đầu' vào attention_maps_overlay_no_title/Base_Image_Che đầu.png
Đã lưu ảnh cơ sở 'Che thân' vào attention_maps_overlay_no_title/Base_Image_Che thân.png
Đã lưu 21 hình ảnh overlay (không tiêu đề) vào thư mục: attention_maps_overlay_no_title/
Đã tạo file nén 'attention_maps_overlay_no_title.zip' chứa các hình ảnh overlay không tiêu đề.
Bạn có thể tải file này xuống.
Đã xóa thư mục tạm thời: attention_maps_overlay_no_title
